# Joint Estimation of GARCH Option Pricing Model

**Author:** Jakob Troger  
**Course:** Financial Econometrics, University of Vienna  
**Date:** January 2026

---

This notebook implements the joint estimation of the Heston-Nandi GARCH option pricing model using S&P 500 return and implied volatility data (2007–2025). It reproduces all results and figures from the accompanying paper.

### Structure
1. **Part 1 — Core Model & Estimation:** Numba-optimized GARCH functions, data loading, MLE and joint estimation
2. **Part 2 — Forensic Analysis:** Counterfactual experiments explaining *why* the model fails to generate realistic implied volatility skew
3. **Part 3 — Run Analysis:** Execute the full forensic analysis pipeline

### Requirements
```
pip install yfinance pandas numpy scipy openpyxl numba matplotlib
```

> **Data note:** S&P 500 returns are downloaded via `yfinance`. Implied volatility surface data must be provided as an Excel file — see `load_iv_surface()` for the expected column format (`SPX XX% MONEYNESS - IMPLIED VOL X MTH`).


## Part 1 — Core Model & Estimation

### 1.1 Imports & Setup

All Numba-JIT compiled functions for the variance path, log-likelihood, characteristic function, and FFT option pricer are defined here. The data loading utilities for S&P 500 returns and the implied volatility surface are also included.

**Key functions:**
- `compute_variance_path_single` — Heston-Nandi GARCH(1,1) variance recursion
- `characteristic_function_single` — Closed-form CF under Q (risk-neutral measure)
- `price_call_fft_single` — European call price via Carr-Madan FFT
- `load_returns` / `load_iv_surface` — Data ingestion


In [ ]:
"""
Joint Estimation of Heston-Nandi GARCH Option Pricing Model

Analysis of the single-factor GARCH model for S&P 500 options

This implements:
1. Joint estimation using returns and IV data
2. IV fit analysis across moneyness and maturity
3. Forensic analysis of why the model fails to capture skew

Requirements:
    pip install yfinance pandas numpy scipy openpyxl numba
"""

import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
import re
import warnings
warnings.filterwarnings('ignore')

from numba import jit

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except ImportError:
    YFINANCE_AVAILABLE = False
    print("Warning: yfinance not installed. Run: pip install yfinance")


# NUMBA-OPTIMIZED FUNCTIONS: SINGLE-FACTOR (Heston-Nandi)


@jit(nopython=True)
def compute_variance_path_single(returns, mu, omega, beta, alpha, gamma):
    """Single-factor GARCH variance path."""
    T = len(returns)
    h = np.zeros(T + 1)
    
    denom = 1 - beta - alpha * gamma * gamma
    if denom > 0:
        h[0] = omega / denom
    else:
        h[0] = 0.0001
    
    if h[0] <= 0 or np.isnan(h[0]):
        mean_r = 0.0
        for i in range(T):
            mean_r += returns[i]
        mean_r /= T
        var_r = 0.0
        for i in range(T):
            var_r += (returns[i] - mean_r) ** 2
        h[0] = var_r / T
    
    for t in range(T):
        if h[t] > 0:
            sqrt_h = np.sqrt(h[t])
            z_t = (returns[t] - mu) / sqrt_h
        else:
            z_t = 0.0
            sqrt_h = 0.0
        h[t+1] = omega + beta * h[t] + alpha * (z_t - gamma * sqrt_h) ** 2
        if h[t+1] < 1e-10:
            h[t+1] = 1e-10
    
    return h


@jit(nopython=True)
def returns_log_likelihood_single(returns, mu, h):
    """Single-factor log-likelihood."""
    T = len(returns)
    ll = 0.0
    log_2pi = np.log(2 * np.pi)
    
    for t in range(T):
        if h[t] <= 0:
            return -1e10
        ll += -0.5 * log_2pi - 0.5 * np.log(h[t]) - 0.5 * (returns[t] - mu) ** 2 / h[t]
    
    return ll


@jit(nopython=True)
def characteristic_function_single(u_real, u_imag, h_t, tau, omega, beta, alpha, gamma_star, r_f):
    """Single-factor characteristic function."""
    A_real = 0.0
    A_imag = 0.0
    B_real = 0.0
    B_imag = 0.0
    
    for _ in range(int(tau)):
        denom_real = 1 - 2 * alpha * B_real
        denom_imag = -2 * alpha * B_imag
        denom_mag_sq = denom_real * denom_real + denom_imag * denom_imag
        
        if denom_mag_sq < 1e-20:
            return np.nan, np.nan
        
        denom_mag = np.sqrt(denom_mag_sq)
        log_denom_real = np.log(denom_mag)
        log_denom_imag = np.arctan2(denom_imag, denom_real)
        
        A_new_real = A_real + (-u_imag) * r_f + omega * B_real - 0.5 * log_denom_real
        A_new_imag = A_imag + u_real * r_f + omega * B_imag - 0.5 * log_denom_imag
        
        gs2a = gamma_star * gamma_star * alpha
        num_real = -0.5 - gs2a * B_real
        num_imag = u_real - gs2a * B_imag
        
        div_real = (num_real * denom_real + num_imag * denom_imag) / denom_mag_sq
        div_imag = (num_imag * denom_real - num_real * denom_imag) / denom_mag_sq
        
        B_new_real = div_real + beta * B_real + gs2a
        B_new_imag = div_imag + beta * B_imag
        
        if np.isnan(A_new_real) or np.isnan(B_new_real) or np.abs(B_new_real) > 1e10:
            return np.nan, np.nan
        
        A_real = A_new_real
        A_imag = A_new_imag
        B_real = B_new_real
        B_imag = B_new_imag
    
    exp_arg_real = A_real + B_real * h_t
    exp_arg_imag = A_imag + B_imag * h_t
    
    exp_mag = np.exp(exp_arg_real)
    result_real = exp_mag * np.cos(exp_arg_imag)
    result_imag = exp_mag * np.sin(exp_arg_imag)
    
    return result_real, result_imag


@jit(nopython=True)
def price_call_fft_single(S, K, h_t, tau, omega, beta, alpha, gamma_star, r_f, N, alpha_cm):
    """Single-factor FFT option pricing."""
    eta = 0.25
    lambda_fft = 2 * np.pi / (N * eta)
    b = N * lambda_fft / 2
    
    k = np.zeros(N)
    psi_real = np.zeros(N)
    psi_imag = np.zeros(N)
    
    for j in range(N):
        k[j] = -b + lambda_fft * j
        v_j = eta * j
        
        u_real = v_j
        u_imag = -(alpha_cm + 1)
        
        phi_real, phi_imag = characteristic_function_single(
            u_real, u_imag, h_t, tau, omega, beta, alpha, gamma_star, r_f
        )
        
        if np.isnan(phi_real):
            return np.nan
        
        denom_real = alpha_cm * alpha_cm + alpha_cm - v_j * v_j
        denom_imag = (2 * alpha_cm + 1) * v_j
        denom_mag_sq = denom_real * denom_real + denom_imag * denom_imag
        
        if denom_mag_sq < 1e-20:
            return np.nan
        
        discount = np.exp(-r_f * tau)
        psi_real[j] = discount * (phi_real * denom_real + phi_imag * denom_imag) / denom_mag_sq
        psi_imag[j] = discount * (phi_imag * denom_real - phi_real * denom_imag) / denom_mag_sq
    
    x_real = np.zeros(N)
    x_imag = np.zeros(N)
    for j in range(N):
        v_j = eta * j
        angle = b * v_j
        cos_angle = np.cos(angle)
        sin_angle = np.sin(angle)
        x_real[j] = eta * (cos_angle * psi_real[j] - sin_angle * psi_imag[j])
        x_imag[j] = eta * (sin_angle * psi_real[j] + cos_angle * psi_imag[j])
    
    log_moneyness = np.log(K / S)
    
    if log_moneyness < k[0] or log_moneyness > k[N-1]:
        return np.nan
    
    idx = 0
    for j in range(N):
        if k[j] > log_moneyness:
            idx = j
            break
    if idx == 0:
        idx = 1
    
    call_price_prev = 0.0
    call_price_curr = 0.0
    
    for m in [idx-1, idx]:
        fft_real = 0.0
        for j in range(N):
            angle = 2 * np.pi * m * j / N
            cos_a = np.cos(angle)
            sin_a = np.sin(angle)
            fft_real += x_real[j] * cos_a - x_imag[j] * sin_a
        
        call_price = np.exp(-alpha_cm * k[m]) / np.pi * fft_real
        if m == idx - 1:
            call_price_prev = call_price
        else:
            call_price_curr = call_price
    
    w = (log_moneyness - k[idx-1]) / (k[idx] - k[idx-1])
    call_price = (1 - w) * call_price_prev + w * call_price_curr
    
    return S * call_price



# DATA LOADING

def parse_iv_column_name(col_name):
    pattern = r'SPX (\d+)% MONEYNESS - IMPLIED VOL (\d+) (MTH|YR)'
    match = re.match(pattern, col_name)
    if not match:
        return None
    moneyness = int(match.group(1))
    duration = int(match.group(2))
    unit = match.group(3)
    maturity = float(duration) if unit == 'YR' else duration / 12.0
    return (moneyness, maturity)


def load_iv_surface(filepath):
    df = pd.read_excel(filepath)
    df['date'] = pd.to_datetime(df['Name'])
    df = df.drop(columns=['Name']).set_index('date')
    iv_wide = df.copy()
    
    records = []
    for col in df.columns:
        parsed = parse_iv_column_name(col)
        if parsed is None:
            continue
        moneyness, maturity = parsed
        for date, iv_value in df[col].items():
            if pd.notna(iv_value):
                records.append({
                    'date': date,
                    'moneyness': moneyness,
                    'maturity': maturity,
                    'iv': iv_value / 100.0
                })
    iv_long = pd.DataFrame(records)
    return iv_wide, iv_long


def load_returns(start_date, end_date, ticker="^GSPC"):
    if not YFINANCE_AVAILABLE:
        raise ImportError("yfinance required")
    
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    
    if 'Adj Close' in data.columns:
        price_col = 'Adj Close'
    elif 'Close' in data.columns:
        price_col = 'Close'
    else:
        raise KeyError("Cannot find price column")
    
    prices = data[price_col].to_frame(name='price')
    prices.index.name = 'date'
    prices['log_return'] = np.log(prices['price'] / prices['price'].shift(1))
    return prices.dropna()


def load_all_data(iv_filepath, ticker="^GSPC"):
    print("Loading IV surface data...")
    iv_wide, iv_long = load_iv_surface(iv_filepath)
    print(f"  IV data: {len(iv_wide)} days, {iv_wide.index[0].date()} to {iv_wide.index[-1].date()}")
    
    print("Loading returns from Yahoo Finance...")
    start = (iv_wide.index[0] - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
    end = (iv_wide.index[-1] + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
    returns = load_returns(start, end, ticker)
    print(f"  Returns: {len(returns)} days")
    
    common_dates = sorted(set(iv_wide.index) & set(returns.index))
    aligned_dates = pd.DatetimeIndex(common_dates)
    print(f"  Aligned: {len(aligned_dates)} common dates")
    
    return {
        'returns': returns.loc[aligned_dates],
        'iv_wide': iv_wide.loc[aligned_dates],
        'iv_long': iv_long[iv_long['date'].isin(aligned_dates)],
        'aligned_dates': aligned_dates,
        'prices': returns.loc[aligned_dates, 'price']
    }



# MODEL CLASSES

class HestonNandiGARCH:
    """Single-factor Heston-Nandi GARCH (baseline)."""
    
    def __init__(self):
        self.param_names = ['mu', 'omega', 'beta', 'alpha', 'gamma', 'lambda_']
        self.bounds = [
            (-0.1, 0.1),      # mu
            (1e-10, 1e-4),    # omega
            (0.0, 0.9999),    # beta
            (1e-10, 1e-4),    # alpha
            (-5, 5),      # gamma
            (-10, 10)         # lambda_
        ]
        self.n_params = 6
    
    def unpack_params(self, theta):
        return {
            'mu': theta[0],
            'omega': theta[1],
            'beta': theta[2],
            'alpha': theta[3],
            'gamma': theta[4],
            'lambda_': theta[5]
        }
    
    def compute_variance_path(self, returns, params):
        h = compute_variance_path_single(
            returns, params['mu'], params['omega'],
            params['beta'], params['alpha'], params['gamma']
        )
        return h, None, None  # Return h, q, s (q and s are None for single factor)
    
    def returns_log_likelihood(self, returns, params):
        h, _, _ = self.compute_variance_path(returns, params)
        return returns_log_likelihood_single(returns, params['mu'], h)
    
    def price_call_option(self, S, K, h_t, q_t, s_t, tau, params, r_f=0.0, N=1024, alpha_cm=1.5):
        gamma_star = params['gamma'] + params['lambda_'] + 0.5
        return price_call_fft_single(
            S, K, h_t, tau,
            params['omega'], params['beta'], params['alpha'],
            gamma_star, r_f, N, alpha_cm
        )
    
    def price_to_iv(self, price, S, K, tau, r_f=0.0):
        if price <= 0 or np.isnan(price):
            return np.nan
        
        sigma = 0.2
        for _ in range(100):
            d1 = (np.log(S / K) + (r_f + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
            d2 = d1 - sigma * np.sqrt(tau)
            
            bs_price = S * norm.cdf(d1) - K * np.exp(-r_f * tau) * norm.cdf(d2)
            vega = S * np.sqrt(tau) * norm.pdf(d1)
            
            if vega < 1e-10:
                return np.nan
            
            diff = bs_price - price
            if abs(diff) < 1e-8:
                return sigma
            
            sigma = sigma - diff / vega
            
            if sigma <= 0 or sigma > 5:
                return np.nan
        
        return sigma
    
    def compute_model_iv(self, S, moneyness, maturity_months, h_t, q_t, s_t, params, r_f_annual=0.02):
        K = S * moneyness / 100.0
        tau_days = maturity_months * 21
        tau_years = maturity_months / 12.0
        r_f_daily = r_f_annual / 252.0
        
        call_price = self.price_call_option(S, K, h_t, q_t, s_t, tau_days, params, r_f_daily)
        
        if np.isnan(call_price) or call_price <= 0:
            return np.nan
        
        return self.price_to_iv(call_price, S, K, tau_years, r_f_annual)
    
    def get_persistence(self, params):
        return params['beta'] + params['alpha'] * params['gamma']**2
    
    def get_uncond_vol(self, params):
        pers = self.get_persistence(params)
        if pers < 1:
            return np.sqrt(252 * params['omega'] / (1 - pers))
        return np.nan




# JOINT ESTIMATOR

class JointEstimator:
    """Joint estimation for both single-factor and two-factor models."""
    
    def __init__(self, returns, prices, iv_data, model):
        self.returns = returns
        self.prices = prices
        self.iv_data = iv_data
        self.model = model
        self.iv_subset = self._subsample_iv_data()
        self._eval_count = 0
    
    def _subsample_iv_data(self):
        """ATM-focused subsampling (Fix B)."""
        df = self.iv_data.copy()
        # Focus on near-ATM options for cleaner estimation
        moneyness_keep = [95, 100, 105]  # ATM-focused
        maturity_keep = [1/12, 3/12, 6/12]
        mask = (df['moneyness'].isin(moneyness_keep)) & (df['maturity'].isin(maturity_keep))
        return df[mask].reset_index(drop=True)
    
    def returns_only_objective(self, theta):
        params = self.model.unpack_params(theta)
        
        # Basic parameter checks
        for key, val in params.items():
            if np.isnan(val):
                return 1e10
        
        ll = self.model.returns_log_likelihood(self.returns, params)
        
        if np.isnan(ll) or np.isinf(ll):
            return 1e10
        
        return -ll
    
    def joint_objective(self, theta, weight_iv=1.0):
        """Joint objective with rebalanced IV term (Fix A)."""
        self._eval_count += 1
        if self._eval_count % 10 == 0:
            print(f"  Evaluation {self._eval_count}...")
        
        params = self.model.unpack_params(theta)
        
        # Parameter checks
        for key, val in params.items():
            if np.isnan(val):
                return 1e10
        
        ll_returns = self.model.returns_log_likelihood(self.returns, params)
        if np.isnan(ll_returns) or np.isinf(ll_returns):
            return 1e10
        
        h, q, s = self.model.compute_variance_path(self.returns, params)
        
        dates = self.iv_subset['date'].unique()
        
        iv_sse = 0.0
        n_iv = 0
        
        for date in dates[::20]:
            try:
                idx = np.where(self.prices.index == date)[0][0]
            except:
                continue
            
            S = self.prices.iloc[idx]
            h_t = h[idx]
            q_t = q[idx] if q is not None else None
            s_t = s[idx] if s is not None else None
            
            day_iv = self.iv_subset[self.iv_subset['date'] == date]
            
            for _, row in day_iv.iterrows():
                model_iv = self.model.compute_model_iv(
                    S, row['moneyness'], row['maturity'] * 12,
                    h_t, q_t, s_t, params
                )
                
                if not np.isnan(model_iv):
                    iv_sse += (model_iv - row['iv'])**2
                    n_iv += 1
        
        if n_iv == 0:
            return 1e10
        
        # REBALANCED OBJECTIVE (Fix A): removed n_iv multiplier
        objective = -ll_returns + weight_iv * iv_sse * 1000  # Scale factor for balance
        
        if self._eval_count % 10 == 0:
            print(f"    n_iv={n_iv}, iv_sse={iv_sse:.3f}, obj={objective:.0f}")
        
        return objective
    
    def estimate_returns_only(self, x0=None):
        print(f"Estimating {self.model.__class__.__name__} from returns only...")
        
        if x0 is None:
            if self.model.n_params == 6:
                x0 = [0.0001, 1e-6, 0.8, 1e-6, 100, 0.0]
            else:  # 10 params
                x0 = [0.0001, 1e-6, 0.95, 1e-7, 50, 1e-7, 0.5, 1e-6, 50, 0.0]
        
        result = minimize(
            self.returns_only_objective,
            x0,
            method='L-BFGS-B',
            bounds=self.model.bounds,
            options={'maxiter': 1000, 'disp': False}
        )
        
        params = self.model.unpack_params(result.x)
        h, q, s = self.model.compute_variance_path(self.returns, params)
        
        print("  Converged:", result.success)
        print("  Log-likelihood:", -result.fun)
        
        return {
            'params': params,
            'theta': result.x,
            'log_likelihood': -result.fun,
            'converged': result.success,
            'variance_path': h,
            'q_path': q,
            's_path': s
        }
    
    def estimate_joint(self, x0=None, weight_iv=1.0):
        print(f"Estimating {self.model.__class__.__name__} jointly...")
        print(f"  Using {len(self.iv_subset)} IV observations (ATM-focused)")
        
        self._eval_count = 0
        
        if x0 is None:
            if self.model.n_params == 6:
                x0 = [0.0001, 1e-6, 0.8, 1e-6, 100, 0.0]
            else:
                x0 = [0.0001, 1e-6, 0.95, 1e-7, 50, 1e-7, 0.5, 1e-6, 50, 0.0]
        
        result = minimize(
            lambda theta: self.joint_objective(theta, weight_iv),
            x0,
            method='L-BFGS-B',
            bounds=self.model.bounds,
            options={'maxiter': 1000, 'disp': False}
        )
        
        params = self.model.unpack_params(result.x)
        h, q, s = self.model.compute_variance_path(self.returns, params)
        
        print("  Converged:", result.success)
        print(f"  Total evaluations: {self._eval_count}")
        
        return {
            'params': params,
            'theta': result.x,
            'converged': result.success,
            'variance_path': h,
            'q_path': q,
            's_path': s
        }


# RESULTS AND ANALYSIS

def print_single_factor_results(results_returns, results_joint, model):
    print("\n" + "="*70)
    print("SINGLE-FACTOR HESTON-NANDI RESULTS")
    print("="*70)
    
    print("\n{:<15} {:>20} {:>20}".format("Parameter", "Returns Only", "Joint"))
    print("-"*55)
    
    for param in model.param_names:
        val_r = results_returns['params'][param]
        val_j = results_joint['params'][param]
        
        if 'omega' in param or 'alpha' in param or 'phi' in param:
            print("{:<15} {:>20.2e} {:>20.2e}".format(param, val_r, val_j))
        else:
            print("{:<15} {:>20.6f} {:>20.6f}".format(param, val_r, val_j))
    
    print("-"*55)
    
    pers_r = model.get_persistence(results_returns['params'])
    pers_j = model.get_persistence(results_joint['params'])
    vol_r = model.get_uncond_vol(results_returns['params'])
    vol_j = model.get_uncond_vol(results_joint['params'])
    
    print("{:<15} {:>20.4f} {:>20.4f}".format("Persistence", pers_r, pers_j))
    print("{:<15} {:>19.2f}% {:>19.2f}%".format("Uncond. Vol", vol_r*100, vol_j*100))



def compute_iv_fit(data, results, model, sample_dates=50):
    params = results['params']
    h = results['variance_path']
    q = results.get('q_path')
    s = results.get('s_path')
    prices = data['prices']
    iv_long = data['iv_long']
    
    unique_dates = iv_long['date'].unique()
    sample_idx = np.linspace(0, len(unique_dates)-1, sample_dates, dtype=int)
    sampled_dates = unique_dates[sample_idx]
    
    records = []
    
    for date in sampled_dates:
        try:
            idx = np.where(prices.index == date)[0][0]
        except:
            continue
        
        S = prices.iloc[idx]
        h_t = h[idx]
        q_t = q[idx] if q is not None else None
        s_t = s[idx] if s is not None else None
        
        day_iv = iv_long[iv_long['date'] == date]
        
        for _, row in day_iv.iterrows():
            model_iv = model.compute_model_iv(
                S, row['moneyness'], row['maturity'] * 12,
                h_t, q_t, s_t, params
            )
            
            if not np.isnan(model_iv):
                records.append({
                    'date': date,
                    'moneyness': row['moneyness'],
                    'maturity': row['maturity'],
                    'observed_iv': row['iv'],
                    'model_iv': model_iv,
                    'error': model_iv - row['iv']
                })
    
    return pd.DataFrame(records)


def print_iv_fit_summary(iv_fit, model_name):
    print(f"\n{'='*70}")
    print(f"IV FIT: {model_name}")
    print("="*70)
    
    if len(iv_fit) == 0:
        print("No valid IV fits computed.")
        return
    
    rmse = np.sqrt((iv_fit['error']**2).mean())
    mae = np.abs(iv_fit['error']).mean()
    
    print(f"\nOverall RMSE: {rmse*100:.2f}%")
    print(f"Overall MAE:  {mae*100:.2f}%")
    
    print("\nRMSE by Moneyness:")
    for m in sorted(iv_fit['moneyness'].unique()):
        subset = iv_fit[iv_fit['moneyness'] == m]
        rmse_m = np.sqrt((subset['error']**2).mean())
        print(f"  {m}%: {rmse_m*100:.2f}%")
    
    print("\nRMSE by Maturity:")
    for tau in sorted(iv_fit['maturity'].unique()):
        subset = iv_fit[iv_fit['maturity'] == tau]
        rmse_t = np.sqrt((subset['error']**2).mean())
        print(f"  {tau*12:.0f}M: {rmse_t*100:.2f}%")



# MAIN EXECUTION

if __name__ == "__main__":
    
    import os
    os.chdir(r"C:\Users\jtrog\OneDrive\Desktop\Uni Wien\Applied Econ\WS25\Financial Econometrics\Project")
    print("Current folder:", os.getcwd())
    
    print("\n" + "="*70)
    print("JOINT ESTIMATION OF GARCH OPTION PRICING MODELS")
    print("Heston-Nandi GARCH Model")
    print("="*70)
    
    # Load data
    IV_FILE = "SP500_IV_SURFACE.xlsx"
    data = load_all_data(IV_FILE)
    
    returns = data['returns']['log_return'].values
    prices = data['prices']
    iv_long = data['iv_long']
    
    print(f"\nData summary:")
    print(f"  Returns: {len(returns)} observations")
    print(f"  Mean daily return: {returns.mean()*100:.4f}%")
    print(f"  Daily volatility: {returns.std()*100:.2f}%")
    print(f"  Annualized volatility: {returns.std()*np.sqrt(252)*100:.1f}%")

    # Summary statistics for Table 1
    from scipy.stats import jarque_bera, skew, kurtosis
    print(f"\n" + "="*70)
    print("SUMMARY STATISTICS (Table 1)")
    print("="*70)
    print(f"  Observations:      {len(returns)}")
    print(f"  Mean (%):          {returns.mean()*100:.4f}")
    print(f"  Std. Dev. (%):     {returns.std()*100:.2f}")
    print(f"  Skewness:          {skew(returns):.2f}")
    print(f"  Excess Kurtosis:   {kurtosis(returns):.1f}")
    print(f"  Minimum (%):       {returns.min()*100:.2f}")
    print(f"  Maximum (%):       {returns.max()*100:.2f}")
    jb_stat, jb_pval = jarque_bera(returns)
    print(f"  Jarque-Bera:       {jb_stat:.0f}{'***' if jb_pval < 0.01 else ''}")

    
    
    # SINGLE-FACTOR MODEL
    
    print("\n" + "="*70)
    print("ESTIMATING SINGLE-FACTOR MODEL (Heston-Nandi)")
    print("="*70)
    
    model_sf = HestonNandiGARCH()
    estimator_sf = JointEstimator(returns, prices, iv_long, model_sf)
    
    print("\nCompiling Numba functions...")
    results_sf_returns = estimator_sf.estimate_returns_only()
    results_sf_joint = estimator_sf.estimate_joint(x0=results_sf_returns['theta'])
    
    print_single_factor_results(results_sf_returns, results_sf_joint, model_sf)

    
    # IV FIT ANALYSIS
    
    print("\n" + "="*70)
    print("COMPUTING IV FIT (Table 3)")
    print("="*70)
    
    print("\nComputing single-factor IV fit...")
    sf_iv_fit = compute_iv_fit(data, results_sf_joint, model_sf)
    print_iv_fit_summary(sf_iv_fit, "Single-Factor Heston-Nandi")

    
    

    # Setup variables for forensic analysis
    params = results_sf_joint['params']
    h0 = results_sf_joint['variance_path'][-1]  # Current variance
    S0 = prices.iloc[-1]  # Current price
    print(f"\nForensic analysis variables set:")
    print(f"  S0 = {S0:.2f}")
    print(f"  h0 = {h0:.6f} (vol = {np.sqrt(252*h0)*100:.1f}%)")
    print(f"  gamma = {params['gamma']:.2f}")
    print(f"  alpha*gamma = {params['alpha']*params['gamma']:.2e}")

    print("\n" + "="*70)
    print("ESTIMATION COMPLETE")
    print("="*70)

## Part 2 — Forensic Analysis

This section mechanically explains why the Heston-Nandi GARCH model fails to generate realistic implied volatility skew. The core finding is a **fundamental tension**: return dynamics require a small shock-impact parameter $\alpha$, but option pricing requires a large $\alpha\gamma$ product for leverage strength.

**Experiments:**
1. `analytical_correlation` — Analytical derivation of $\text{Corr}(r_t, h_{t+1})$
2. `experiment_correlation_vs_gamma` — How correlation varies with $\gamma$ (Figure 1 in paper)
3. `experiment_alpha_gamma_interaction` — Contour plot of the $\alpha$-$\gamma$ interaction (Figure 2)
4. `experiment_variance_ratio` — Variance asymmetry after negative vs. positive returns (Figure 3)
5. `experiment_iv_smile_vs_gamma` — Implied volatility smile for different $\gamma$ (Figure 4)
6. `experiment_maturity_fit` — Model vs. market IV across maturities (Figure 5)


In [ ]:
"""
Forensic Analysis of GARCH Option Pricing Failure
==================================================

This code performs deep mechanical analysis of why the Heston-Nandi GARCH model
fails to generate realistic implied volatility skew.

Experiments:
1. Analytical derivation of corr(r_t, h_{t+1})
2. Simulation: How does skew change with γ?
3. Simulation: IV smile as function of γ
4. Why short-maturity options are impossible to fit
5. Parameter interaction analysis (α, β, γ)

Run this AFTER your main estimation to generate figures and analysis for the paper.
"""

import numpy as np
import matplotlib.pyplot as plt
from numba import jit
from scipy.stats import norm
from scipy.optimize import brentq

# Set publication-quality style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['figure.dpi'] = 150



# 1. ANALYTICAL DERIVATION OF CORR(r_t, h_{t+1})


def analytical_correlation(alpha, beta, gamma, omega, h_bar=None):
    """
    Derive the correlation between r_t and h_{t+1} analytically.
    
    From the GARCH equation:
    h_{t+1} = ω + βh_t + α(z_t - γ√h_t)²
            = ω + βh_t + α(z_t² - 2γ√h_t·z_t + γ²h_t)
    
    The return is: r_t = μ + λh_t + √h_t·z_t
    
    The covariance comes from the cross-term: -2αγ√h_t·z_t
    
    Cov(r_t, h_{t+1}) = Cov(√h_t·z_t, -2αγ√h_t·z_t)
                      = -2αγ · E[h_t · z_t²]
                      = -2αγ · E[h_t] · E[z_t²]  (independence)
                      = -2αγ · h̄ · 1
                      = -2αγh̄
    
    Var(r_t) ≈ h̄ (dominated by √h_t·z_t term)
    
    Var(h_{t+1}) is more complex, but approximately:
    Var(h_{t+1}) ≈ 2α²(1 + 2γ²h̄)h̄ + ... (simplified)
    
    For a rough approximation:
    Corr(r_t, h_{t+1}) ≈ -2αγh̄ / (√h̄ · σ_h)
    
    A simpler approximation from the literature:
    The key insight is that the leverage effect strength is proportional to αγ.
    """
    if h_bar is None:
        # Compute unconditional variance
        persistence = beta + alpha * gamma**2
        if persistence >= 1:
            return np.nan
        h_bar = (omega + alpha) / (1 - persistence)
    
    # The covariance term is -2αγh̄
    cov_r_h = -2 * alpha * gamma * h_bar
    
    # Variance of return (approximately h̄)
    var_r = h_bar
    
    # Variance of h_{t+1} (approximation)
    # This is complex, but scales with α²
    var_h = 2 * alpha**2 * h_bar * (1 + 2 * gamma**2 * h_bar)
    
    # Correlation
    if var_r <= 0 or var_h <= 0:
        return np.nan
    
    corr = cov_r_h / np.sqrt(var_r * var_h)
    
    return corr, cov_r_h, h_bar


def print_analytical_derivation():
    """Print the analytical derivation for the paper."""
    print("="*70)
    print("ANALYTICAL DERIVATION: Corr(r_t, h_{t+1})")
    print("="*70)
    
    print("""
From the Heston-Nandi GARCH model:

    r_t = μ + λh_t + √h_t · z_t
    h_{t+1} = ω + βh_t + α(z_t - γ√h_t)²

Expanding the variance equation:

    h_{t+1} = ω + βh_t + α[z_t² - 2γ√h_t·z_t + γ²h_t]

The covariance between r_t and h_{t+1} comes from the cross-term:

    Cov(r_t, h_{t+1}) = Cov(√h_t·z_t, -2αγ√h_t·z_t)
                      = -2αγ · E[h_t·z_t²]
                      = -2αγ · E[h_t] · E[z_t²]
                      = -2αγ · h̄

Key insight: The covariance is proportional to the product αγ.

With your estimated parameters:
    α = 7.31 × 10⁻⁶
    γ = 5.00
    αγ = 3.66 × 10⁻⁵

This product is TINY, explaining why the leverage effect is so weak.
""")


# 2. SIMULATION: CORRELATION AS FUNCTION OF γ


@jit(nopython=True)
def simulate_correlation(omega, beta, alpha, gamma, n_sim=50000, seed=42):
    """Simulate returns and variance to compute correlation."""
    np.random.seed(seed)
    
    # Initialize at unconditional variance
    persistence = beta + alpha * gamma**2
    if persistence >= 1:
        h_bar = omega / (1 - beta)  # Fallback
    else:
        h_bar = (omega + alpha) / (1 - persistence)
    
    h = h_bar
    
    returns = np.zeros(n_sim)
    variances = np.zeros(n_sim)
    
    for t in range(n_sim):
        z = np.random.randn()
        r = -0.5 * h + np.sqrt(h) * z  # Simplified return (no μ, λ)
        h_next = omega + beta * h + alpha * (z - gamma * np.sqrt(h))**2
        h_next = max(h_next, 1e-12)
        
        returns[t] = r
        variances[t] = h_next
        h = h_next
    
    # Compute correlation between r_t and h_{t+1}
    corr = np.corrcoef(returns[:-1], variances[1:])[0, 1]
    
    return corr


def experiment_correlation_vs_gamma(omega, beta, alpha, gamma_range=None, save_path=None):
    """
    Experiment 1: How does corr(r_t, h_{t+1}) change with γ?
    
    This shows that even with extreme γ, the correlation remains weak
    because α is so small.
    """
    if gamma_range is None:
        gamma_range = np.linspace(0, 20, 21)
    
    correlations = []
    for gamma in gamma_range:
        corr = simulate_correlation(omega, beta, alpha, gamma)
        correlations.append(corr)
    
    # Also compute what γ would be needed for corr = -0.3
    target_corr = -0.3
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(gamma_range, correlations, 'b-o', linewidth=2, markersize=6)
    ax.axhline(y=-0.3, color='red', linestyle='--', linewidth=1.5, label='Target for realistic skew (-0.3)')
    ax.axhline(y=-0.5, color='red', linestyle=':', linewidth=1.5, label='Strong skew (-0.5)')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    ax.axvline(x=5, color='green', linestyle='--', linewidth=1.5, label='Estimated γ = 5 (at bound)')
    
    ax.set_xlabel('Leverage Parameter (γ)')
    ax.set_ylabel('Correlation(r_t, h_{t+1})')
    ax.set_title('Why Increasing γ Alone Cannot Generate Realistic Skew\n' + 
                 f'(Fixed α = {alpha:.2e}, β = {beta:.4f})')
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)
    
    # Add annotation
    ax.annotate(f'At γ=5: corr = {correlations[5]:.3f}', 
                xy=(5, correlations[5]), xytext=(8, correlations[5] + 0.02),
                arrowprops=dict(arrowstyle='->', color='green'),
                fontsize=10, color='green')
    
    ax.annotate(f'At γ=20: corr = {correlations[-1]:.3f}\n(still far from -0.3)', 
                xy=(20, correlations[-1]), xytext=(15, -0.15),
                arrowprops=dict(arrowstyle='->', color='blue'),
                fontsize=10, color='blue')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return gamma_range, correlations


# 3. WHAT PARAMETERS WOULD GENERATE REALISTIC SKEW?


def experiment_alpha_gamma_interaction(beta, omega, save_path=None):
    """
    Experiment 2: Show the α-γ interaction.
    
    Plot contours of corr(r_t, h_{t+1}) as function of both α and γ.
    Show that you need BOTH large γ AND large α to get realistic skew.
    """
    alpha_range = np.logspace(-7, -3, 30)  # 1e-7 to 1e-3
    gamma_range = np.linspace(0, 15, 30)
    
    corr_matrix = np.zeros((len(alpha_range), len(gamma_range)))
    
    for i, alpha in enumerate(alpha_range):
        for j, gamma in enumerate(gamma_range):
            try:
                corr = simulate_correlation(omega, beta, alpha, gamma, n_sim=10000)
                corr_matrix[i, j] = corr
            except:
                corr_matrix[i, j] = np.nan
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create contour plot
    G, A = np.meshgrid(gamma_range, alpha_range)
    
    levels = [-0.5, -0.4, -0.3, -0.2, -0.1, -0.05, -0.02, 0]
    cs = ax.contour(G, A, corr_matrix, levels=levels, colors='blue')
    ax.clabel(cs, inline=True, fontsize=9, fmt='%.2f')
    
    # Fill regions
    cf = ax.contourf(G, A, corr_matrix, levels=20, cmap='RdYlBu', alpha=0.7)
    plt.colorbar(cf, ax=ax, label='Correlation(r_t, h_{t+1})')
    
    # Mark estimated parameters
    ax.plot(5, 7.31e-6, 'r*', markersize=20, label=f'Estimated (γ=5, α=7.3e-6)')
    
    # Mark region needed for realistic skew
    ax.axhline(y=1e-4, color='green', linestyle='--', label='α = 1e-4 (10x larger needed)')
    
    ax.set_xlabel('Leverage Parameter (γ)')
    ax.set_ylabel('Shock Impact (α)')
    ax.set_yscale('log')
    ax.set_title('Parameter Interaction: Both α AND γ Must Be Large for Skew\n' +
                 'Contours show Corr(r_t, h_{t+1})')
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return alpha_range, gamma_range, corr_matrix



# 4. IV SMILE AS FUNCTION OF γ (using Monte Carlo)

@jit(nopython=True)
def mc_option_price(S0, K, h0, tau, omega, beta, alpha, gamma_star, r_f, n_paths=20000):
    """Monte Carlo option pricing under Q."""
    np.random.seed(42)
    
    payoffs = np.zeros(n_paths)
    
    for i in range(n_paths):
        S = S0
        h = h0
        
        for t in range(int(tau)):
            z = np.random.randn()
            log_ret = r_f - 0.5*h + np.sqrt(max(h, 1e-12))*z
            S = S * np.exp(log_ret)
            h = omega + beta*h + alpha*(z - gamma_star*np.sqrt(max(h, 1e-12)))**2
            h = max(h, 1e-12)
        
        payoffs[i] = max(S - K, 0)
    
    return np.exp(-r_f * tau) * np.mean(payoffs)


def bs_call(S, K, T, r, sigma):
    """Black-Scholes call price."""
    if sigma <= 0 or T <= 0:
        return max(S - K * np.exp(-r * T), 0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_vol(S, K, T, r, price):
    """Invert BS to get IV."""
    if price <= max(S - K * np.exp(-r * T), 1e-10):
        return np.nan
    try:
        return brentq(lambda sig: bs_call(S, K, T, r, sig) - price, 0.001, 2.0)
    except:
        return np.nan


def experiment_iv_smile_vs_gamma(omega, beta, alpha, lambda_, h0, S0=100, save_path=None):
    """
    Experiment 3: Plot IV smile for different values of γ.
    
    Shows that even with γ = 20, the smile remains nearly flat.
    """
    tau = 63  # 3 months
    r_f = 0.02 / 252
    T_years = tau / 252
    r_annual = r_f * 252
    
    moneyness_range = [90, 95, 100, 105, 110]
    gamma_values = [1, 5, 10, 20]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['blue', 'green', 'orange', 'red']
    
    for gamma, color in zip(gamma_values, colors):
        gamma_star = gamma + lambda_ + 0.5
        
        ivs = []
        for m in moneyness_range:
            K = S0 * m / 100
            price = mc_option_price(S0, K, h0, tau, omega, beta, alpha, gamma_star, r_f)
            iv = implied_vol(S0, K, T_years, r_annual, price)
            ivs.append(iv * 100 if iv and not np.isnan(iv) else np.nan)
        
        ax.plot(moneyness_range, ivs, 'o-', color=color, linewidth=2, 
                markersize=8, label=f'γ = {gamma}')
    
    # Add typical market skew for comparison
    market_skew = [18, 15, 13, 12, 11.5]  # Typical SPX skew
    ax.plot(moneyness_range, market_skew, 'k--', linewidth=2, 
            markersize=8, label='Typical Market Skew')
    
    ax.set_xlabel('Moneyness (%)')
    ax.set_ylabel('Implied Volatility (%)')
    ax.set_title('IV Smile for Different Leverage Parameters\n' +
                 'Model smile remains flat regardless of γ')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Add annotation
    ax.annotate('Market has steep downward skew', 
                xy=(92, 17), fontsize=10, color='black')
    ax.annotate('Model smiles are nearly flat', 
                xy=(92, 12), fontsize=10, color='blue')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
# 5. WHY SHORT-MATURITY OPTIONS ARE Difficult TO FIT


def experiment_maturity_effect(omega, beta, alpha, gamma, lambda_, h0, S0=100, save_path=None):
    """
    Experiment 4: Show IV error by maturity.
    
    Short-maturity options require jump risk to fit.
    GARCH diffusion cannot capture the steep short-term skew.
    """
    maturities = [21, 63, 126, 252]  # 1M, 3M, 6M, 12M
    maturity_labels = ['1M', '3M', '6M', '12M']
    
    r_f = 0.02 / 252
    gamma_star = gamma + lambda_ + 0.5
    
    moneyness_range = [90, 95, 100, 105, 110]
    
    # Typical market IVs (illustrative)
    market_ivs = {
        21: [25, 18, 14, 12, 11],   # 1M: steep skew
        63: [20, 16, 14, 13, 12],   # 3M: moderate skew
        126: [18, 15, 14, 13, 12.5], # 6M: flatter
        252: [17, 15, 14, 13.5, 13]  # 12M: flattest
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, (tau, label) in enumerate(zip(maturities, maturity_labels)):
        ax = axes[idx]
        T_years = tau / 252
        r_annual = r_f * 252
        
        model_ivs = []
        for m in moneyness_range:
            K = S0 * m / 100
            price = mc_option_price(S0, K, h0, tau, omega, beta, alpha, gamma_star, r_f)
            iv = implied_vol(S0, K, T_years, r_annual, price)
            model_ivs.append(iv * 100 if iv and not np.isnan(iv) else np.nan)
        
        ax.plot(moneyness_range, market_ivs[tau], 'ko-', linewidth=2, 
                markersize=8, label='Market IV', markerfacecolor='white')
        ax.plot(moneyness_range, model_ivs, 'b^--', linewidth=2, 
                markersize=8, label='Model IV')
        
        # Compute RMSE
        valid = [(m, mo) for m, mo in zip(market_ivs[tau], model_ivs) if not np.isnan(mo)]
        if valid:
            rmse = np.sqrt(np.mean([(m - mo)**2 for m, mo in valid]))
            ax.set_title(f'{label} Maturity (RMSE = {rmse:.1f}%)')
        else:
            ax.set_title(f'{label} Maturity')
        
        ax.set_xlabel('Moneyness (%)')
        ax.set_ylabel('Implied Volatility (%)')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(5, 30)
    
    plt.suptitle('Why Short-Maturity Options Are Hardest to Fit\n' +
                 'Short maturities have steepest skew (jump-dominated)', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()


# 6. MODEL IV VS MARKET IV (SINGLE FIGURE FOR PAPER)


def plot_model_vs_market_smile(model_ivs, market_ivs, moneyness, date_str, save_path=None):
    """
    Create the key figure: Model IV smile vs Market IV smile.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(moneyness, market_ivs, 'ko-', linewidth=2.5, markersize=10, 
            label='Market IV', markerfacecolor='white', markeredgewidth=2)
    ax.plot(moneyness, model_ivs, 'b^--', linewidth=2, markersize=9, 
            label='Heston-Nandi GARCH', alpha=0.8)
    
    ax.set_xlabel('Moneyness (%)')
    ax.set_ylabel('Implied Volatility (%)')
    ax.set_title(f'Model vs Market Implied Volatility Smile')
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Add annotation explaining the gap
    ax.annotate('Market: steep negative skew\n(crash protection premium)', 
                xy=(92, market_ivs[1]), xytext=(85, market_ivs[1] + 5),
                fontsize=10, ha='center',
                arrowprops=dict(arrowstyle='->', color='black'))
    
    ax.annotate('Model: nearly flat\n(weak leverage effect)', 
                xy=(92, model_ivs[1]), xytext=(85, model_ivs[1] - 5),
                fontsize=10, ha='center', color='blue',
                arrowprops=dict(arrowstyle='->', color='blue'))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()


# 7. VARIANCE RATIO ANALYSIS


@jit(nopython=True)
def simulate_variance_ratio(omega, beta, alpha, gamma, n_sim=50000):
    """
    Compute variance after negative vs positive returns.
    
    For realistic skew, variance after negative returns should be
    1.5-2x higher than after positive returns.
    """
    np.random.seed(42)
    
    persistence = beta + alpha * gamma**2
    if persistence >= 1:
        h_bar = omega / (1 - beta)
    else:
        h_bar = (omega + alpha) / (1 - persistence)
    
    h = h_bar
    
    var_after_neg = []
    var_after_pos = []
    
    for t in range(n_sim):
        z = np.random.randn()
        r = -0.5 * h + np.sqrt(h) * z
        h_next = omega + beta * h + alpha * (z - gamma * np.sqrt(h))**2
        h_next = max(h_next, 1e-12)
        
        if r < 0:
            var_after_neg.append(h_next)
        else:
            var_after_pos.append(h_next)
        
        h = h_next
    
    mean_neg = np.mean(np.array(var_after_neg))
    mean_pos = np.mean(np.array(var_after_pos))
    
    return mean_neg / mean_pos


def experiment_variance_ratio_vs_gamma(omega, beta, alpha, save_path=None):
    """Show variance ratio as function of γ."""
    gamma_range = np.linspace(0, 20, 21)
    
    ratios = []
    for gamma in gamma_range:
        ratio = simulate_variance_ratio(omega, beta, alpha, gamma)
        ratios.append(ratio)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(gamma_range, ratios, 'b-o', linewidth=2, markersize=6)
    ax.axhline(y=1.5, color='red', linestyle='--', label='Target ratio (1.5x)')
    ax.axhline(y=2.0, color='red', linestyle=':', label='Strong asymmetry (2.0x)')
    ax.axhline(y=1.0, color='gray', linestyle='-', linewidth=0.5)
    ax.axvline(x=5, color='green', linestyle='--', label='Estimated γ = 5')
    
    ax.set_xlabel('Leverage Parameter (γ)')
    ax.set_ylabel('Variance Ratio (after negative / after positive return)')
    ax.set_title('Variance Asymmetry: How Much Higher is Variance After Negative Returns?')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Annotation
    ax.annotate(f'At γ=5: ratio = {ratios[5]:.2f}\n(nearly symmetric)', 
                xy=(5, ratios[5]), xytext=(8, ratios[5] + 0.1),
                arrowprops=dict(arrowstyle='->', color='green'),
                fontsize=10)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return gamma_range, ratios


# MAIN: RUN ALL FORENSIC EXPERIMENTS


def run_forensic_analysis(params, h0, S0=2000, save_figures=True):
    """
    Run all forensic experiments with your estimated parameters.
    
    Parameters:
    -----------
    params : dict with keys 'omega', 'beta', 'alpha', 'gamma', 'lambda_'
    h0 : float, current variance estimate
    S0 : float, current stock price
    save_figures : bool, whether to save figures
    """
    
    omega = params['omega']
    beta = params['beta']
    alpha = params['alpha']
    gamma = params['gamma']
    lambda_ = params['lambda_']
    
    print("="*70)
    print("FORENSIC ANALYSIS OF GARCH OPTION PRICING FAILURE")
    print("="*70)
    
    # 1. Analytical derivation
    print("\n" + "="*70)
    print("1. ANALYTICAL DERIVATION")
    print("="*70)
    print_analytical_derivation()
    
    result = analytical_correlation(alpha, beta, gamma, omega)
    if result:
        corr_analytical, cov, h_bar = result
        print(f"\nWith your parameters:")
        print(f"  α = {alpha:.2e}")
        print(f"  γ = {gamma:.2f}")
        print(f"  αγ = {alpha * gamma:.2e}")
        print(f"  Unconditional variance h̄ = {h_bar:.6f}")
        print(f"  Covariance(r, h) = -2αγh̄ = {cov:.2e}")
    
    # 2. Correlation vs gamma
    print("\n" + "="*70)
    print("2. CORRELATION VS LEVERAGE PARAMETER")
    print("="*70)
    gamma_range, correlations = experiment_correlation_vs_gamma(
        omega, beta, alpha,
        save_path='figure_corr_vs_gamma.png' if save_figures else None
    )
    print(f"\nSimulated correlations:")
    for g, c in zip(gamma_range[::5], correlations[::5]):
        print(f"  γ = {g:5.1f}: corr(r, h) = {c:.4f}")
    
    # 3. Alpha-Gamma interaction
    print("\n" + "="*70)
    print("3. PARAMETER INTERACTION (α vs γ)")
    print("="*70)
    experiment_alpha_gamma_interaction(
        beta, omega,
        save_path='figure_alpha_gamma_interaction.png' if save_figures else None
    )
    
    # 4. IV smile vs gamma
    print("\n" + "="*70)
    print("4. IV SMILE FOR DIFFERENT γ VALUES")
    print("="*70)
    experiment_iv_smile_vs_gamma(
        omega, beta, alpha, lambda_, h0, S0,
        save_path='figure_iv_smile_vs_gamma.png' if save_figures else None
    )
    
    # 5. Maturity effect
    print("\n" + "="*70)
    print("5. WHY SHORT-MATURITY OPTIONS ARE HARDEST TO FIT")
    print("="*70)
    experiment_maturity_effect(
        omega, beta, alpha, gamma, lambda_, h0, S0,
        save_path='figure_maturity_effect.png' if save_figures else None
    )
    
    # 6. Variance ratio
    print("\n" + "="*70)
    print("6. VARIANCE ASYMMETRY ANALYSIS")
    print("="*70)
    gamma_range, ratios = experiment_variance_ratio_vs_gamma(
        omega, beta, alpha,
        save_path='figure_variance_ratio.png' if save_figures else None
    )
    print(f"\nVariance ratios (var after neg / var after pos):")
    for g, r in zip(gamma_range[::5], ratios[::5]):
        print(f"  γ = {g:5.1f}: ratio = {r:.3f}")
    
    print("\n" + "="*70)
    print("FORENSIC ANALYSIS COMPLETE")
    print("="*70)
    print("\nKey findings:")
    print(f"  1. Correlation at estimated γ={gamma}: {correlations[int(gamma)]:.4f}")
    print(f"     (Need -0.3 to -0.5 for realistic skew)")
    print(f"  2. Even at γ=20, correlation is only: {correlations[-1]:.4f}")
    print(f"  3. The product αγ = {alpha*gamma:.2e} is too small")
    print(f"  4. Would need α ≈ 1e-4 (10-100x larger) to generate skew")
    print(f"  5. But larger α would break the return fit")
    print(f"\nThis is the fundamental tension: parameters that fit returns")
    print(f"cannot generate the skew needed for option pricing.")



# USAGE


if __name__ == "__main__":
    # Example with typical estimated parameters
    params = {
        'omega': 7.21e-7,
        'beta': 0.8028,
        'alpha': 7.31e-6,
        'gamma': 5.0,
        'lambda_': -0.006
    }
    
    h0 = 0.000076  # Current variance
    S0 = 2000      # Current price
    
    print("Run this after estimation with:")
    print("  run_forensic_analysis(results_sf_joint['params'], h_t, S)")

## Part 3 — Run the Full Analysis

Run this cell after Part 1 and Part 2 are loaded, and after you have run the main estimation to obtain `params`, `h0`, and `S0`.

`run_forensic_analysis(params, h0, S0, save_figures=True)` will sequentially execute all six experiments and save publication-quality figures to disk.


In [ ]:

# Now actually run it
run_forensic_analysis(params, h0, S0, save_figures=True)